# Task 1 EDA: `handicaps` in Link-Lives Release 2 (1801, 1850, 1860, 1880, 1901)

**Scope.** This notebook describes what the data shows. It decides nothing: the Task 1 method, whether to use an LLM for categories, the embedding scope and the File B impact definition are the user's decisions after this EDA (see the closing section "Facts for open decisions").

**Data handling.** Only aggregates, distributions, plots and value-frequency tables are displayed (D-02). No record-level rows are shown: no `pa_id` or other identifier columns, no names next to other attributes, no raw-frame previews. All columns are stored as strings in the parquet files (D-06), so numeric readings below are explicit casts that report their parse failures.

**Reading rule.** An empty `handicaps` value is not evidence of "no handicap". It may mean the question was not asked or the value was not transcribed. Column descriptions below describe what the data shows, not official definitions; the Link-Lives Release 2 Guide is the authority.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display
from rapidfuzz import fuzz, process

# nbconvert runs the kernel in the notebook's directory, so scripts/ is one level up.
sys.path.insert(0, str(Path.cwd().resolve().parent / "scripts"))
import project_paths as pp

pl.Config.set_tbl_rows(60)
pl.Config.set_fmt_str_lengths(80)

YEARS = pp.CENSUS_YEARS
LF = {y: pl.scan_parquet(pp.census_parquet(y)) for y in YEARS}
MIN_EXAMPLE_COUNT = 5
CAPACITY_LINES = (200, 300, 360)
CLUSTER_THRESHOLDS = (80, 85, 90)
REF_THRESHOLD = 85
FACTS = {}


def filled(col):
    # Not null and not blank after strip.
    return pl.col(col).is_not_null() & (pl.col(col).str.strip_chars() != "")


def trivial_norm(expr):
    # lowercase, punctuation runs -> space, collapse whitespace, strip.
    return (
        expr.str.to_lowercase()
        .str.replace_all(r"[^\w\s]+", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def fact(key, value):
    if isinstance(value, float):
        value = round(value, 4)
    FACTS[key] = value
    print(f"FACT {key}={value}")


def safe_div(a, b):
    return a / b if b else 0.0

## 1. `handicaps` per census year

In [ ]:
def year_summary(y):
    row = (
        LF[y]
        .select(
            pl.len().alias("rows_n"),
            filled("handicaps").sum().alias("filled_n"),
            pl.col("handicaps").filter(filled("handicaps")).n_unique().alias("raw_distinct_n"),
        )
        .collect()
        .row(0, named=True)
    )
    row["year"] = y
    row["fill_rate"] = safe_div(row["filled_n"], row["rows_n"])
    return row


summary = pl.DataFrame([year_summary(y) for y in YEARS]).select(
    "year", "rows_n", "filled_n", "fill_rate", "raw_distinct_n"
)
for r in summary.iter_rows(named=True):
    fact(f"task1.rows.{r['year']}", r["rows_n"])
    fact(f"task1.fill_rate.{r['year']}", r["fill_rate"])
    fact(f"task1.raw_distinct.{r['year']}", r["raw_distinct_n"])
summary

**Caveat next to these numbers.** "Missing" here means null or blank after stripping whitespace. An empty `handicaps` value may mean the question was not asked or the value was not transcribed; it must not be read as "no handicap". `raw_distinct_n` counts distinct filled strings exactly as stored. Years with very few filled values are shown as they are, and all rates are zero-safe.

## Facts for open decisions

Every `FACT` line below was computed in this notebook. Later sections are added above this one.

In [ ]:
for k, v in FACTS.items():
    print(f"FACT {k}={v}")